<a href="https://colab.research.google.com/github/Mitch-Cruz/03MAIR-Algoritmos-de-Optimizacion/blob/main/Trabajo%20Pr%C3%A1ctico/Trabajo_Pr%C3%A1ctico_Algoritmos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Algoritmos de optimización - Trabajo Práctico<br>
Nombre y Apellidos: Reinaldo Mitchell Cruz Barrios <br>
Url: https://github.com/Mitch-Cruz/03MAIR-Algoritmos-de-Optimizacion/tree/main/Trabajo%20Pr%C3%A1ctico <br>
Google Colab: https://colab.research.google.com/drive/17LXOhHK1hLXs6C8hN0vwdwbJiKedELut <br>
Problema:
1. Sesiones de doblaje <br>

Descripción del problema:

Se precisa coordinar el doblaje de una película. Los actores del doblaje deben coincidir en las tomas en las que sus personajes aparecen juntos en las diferentes tomas. Los actores de doblaje cobran todos la misma cantidad por cada día que deben desplazarse hasta el estudio de grabación independientemente del número de tomas que se graben. No es posible grabar más de 6 tomas por día. El objetivo es planificar las sesiones por día de manera que el gasto por los servicios de los actores de doblaje sea el menor posible.

Los datos son:

- Número de actores: 10
- Número de tomas : 30

Ver csv file adjunto llamado: Datos problema doblaje(30 tomas, 10 actores) - Hoja 1.csv

O en la web:

https://docs.google.com/spreadsheets/d/1Ipn6IrbQP4ax8zOnivdBIw2lN0JISkJG4fXndYd27U0/edit?gid=0#gid=0


- 1 indica que el actor participa en la toma
- 0 en caso contrario
                                        

#Modelo
- *¿Como represento el espacio de soluciones?*

**Espacio de soluciones:**

Una solución es una asignación de cada toma a un día.

Representación práctica:

lista de 30 enteros
sol = [d0, d1, d2, ..., d29]          
sol[t] = día (0, 1, 2, 3 o 4) en el que se graba la toma t

Cada día debe tener exactamente 6 tomas (30 tomas ÷ 5 días = 6).

Esta codificación permite:

>Fácil generación de soluciones iniciales

>Cruce y mutación rápidas

>Búsqueda local por intercambio de tomas

- *¿Cual es la función objetivo?*


**Función objetivo:**

Minimizar el coste total de las convocatorias de los actores.
$$f(\text{sol}) = \sum_{a=0}^{9} \left| \left\{ d \mid \text{existe toma } t \text{ en día } d \text{ y actor } a \text{ participa en } t \right\} \right|$$
Es decir: Cada actor cobra un día completo por cada día en el que tiene al menos una toma.

El objetivo es reducir el número total de “días-actor”.


- *¿Como implemento las restricciones?*

**Restricciones:**

Restricciones hard (obligatorias):

>Cada toma se graba exactamente una vez.

>Cada día tiene exactamente 6 tomas.

Cómo se garantizan:

>La solución inicial siempre reparte exactamente 6 tomas por día.

>Después de cada cruce y cada mutación se ejecuta la función reparar_balance() que mueve tomas entre días hasta que todos tengan exactamente 6.

>La búsqueda local únicamente realiza intercambios entre días distintos, garantizando que la cardinalidad se mantiene constante, dado que cada día tiene un número fijo de tomas (6).

Sobre el uso necesario de reparar_balance():

>Aunque la búsqueda local preserva la cardinalidad mediante intercambios balanceados, los operadores genéticos (cruce y mutación) pueden generar soluciones inviables. Por este motivo, tras cada cruce y mutación se aplica una función de reparación que restablece la restricción de seis tomas por día.

#Análisis
- *¿Que complejidad tiene el problema?. Orden de complejidad y Contabilizar el espacio de soluciones*

**Complejidad:**

El problema es NP-hard.

Se puede reducir fácilmente del problema Set Partitioning with cardinality constraints o del Minimum Cost Covering with fixed-size bins.

No existe algoritmo polinómico conocido (salvo que P = NP).

Nota para estudiar: NP-hard significa que el problema es al menos tan difícil como los problemas NP-completos.

Ejemplo de reducción: Las tomas son elementos, días son bins de capacidad 6, coste es la unión de actores por bin.

Pregunta: "¿Es NP-completo?" (Respuesta: Sí, porque se puede verificar una solución en tiempo polinómico).

Orden de complejidad y espacio de soluciones

Número de soluciones posibles (días etiquetados 0 a 4, distinguibles):

$$\binom{30}{6} \times \binom{24}{6} \times \binom{18}{6} \times \binom{12}{6} \times \binom{6}{6} = \frac{30!}{(6!)^5} = 1,370,874,167,589,326,400 \approx 1.37 \times 10^{18}$$

Si consideramos los días indistinguibles (particiones sin orden), se divide por 5! = 120 y queda:

 >11,423,951,396,577,720 ≈ 1.14 × 10¹⁶.

Es imposible enumerarlas todas, por eso usamos metaheurística.

**Complejidad temporal** del algoritmo completo: O(G · P · n²)
(dominado por la búsqueda local O(n²)) donde:

>G = número de generaciones (400)

>P = tamaño población (120)

>n = número de tomas (30)


--> Orden O(n²) por iteración efectiva, tiempo total polinómico en el tamaño de la instancia.

#Diseño
- *¿Que técnica utilizo?*

**Técnica utilizada: Algoritmo Genético + Búsqueda Local = Algoritmo Memético**

- *¿Por qué?*

Razones académicas y prácticas:

>Es una de las metaheurísticas más estudiadas en Algoritmos de Optimización (algoritmos evolutivos + mejora local).

>Combina la exploración global (población, cruce, mutación) con la explotación local (búsqueda 2-swap).

>Para este tamaño de instancia (30 tomas) converge en segundos a soluciones de excelente calidad (normalmente 23–25, a veces el óptimo conocido 23).

>Es fácil de explicar, implementar y analizar (se pueden mostrar convergencia, efecto de la búsqueda local, etc.).

>Otras técnicas válidas serían Recocido Simulado o Búsqueda Tabú, pero el memético es el que mejor balance (calidad/claridad) tiene para un trabajo académico como este.

In [54]:
# Solución.

import random
import copy
import time
import numpy as np
import pandas as pd
import requests
import os

In [55]:
# Descargamos el fichero de la web.
url = "https://docs.google.com/spreadsheets/d/1Ipn6IrbQP4ax8zOnivdBIw2lN0JISkJG4fXndYd27U0/export?format=csv&gid=0"
filename = "Datos problema doblaje(30 tomas, 10 actores) - Hoja 1.csv"

print("Descargando datos de doblaje...")

try:
    r = requests.get(url, timeout=10)
    r.raise_for_status()

    with open(filename, 'wb') as f:
        f.write(r.content)

    print(f"¡Descargado correctamente! --> {filename}")
    print(f"Tamaño del archivo: {os.path.getsize(filename):,} bytes")

except Exception as e:
    print("No se pudo descargar el archivo.")
    print("Error:", str(e))

Descargando datos de doblaje...
¡Descargado correctamente! --> Datos problema doblaje(30 tomas, 10 actores) - Hoja 1.csv
Tamaño del archivo: 900 bytes


In [56]:
# Leer csv file.
df = pd.read_csv(filename, header=None)

# Análisis de la estructura del file.
print("Forma del DataFrame:", df.shape)
print("\nNombres exactos de las columnas:")
print(df.columns.tolist())

print("\nPrimeras 5 filas completas:")
print(df.head().to_string(index=False))

print("\nTipos de datos:")
print(df.dtypes)

Forma del DataFrame: (34, 13)

Nombres exactos de las columnas:
[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]

Primeras 5 filas completas:
  0     1   2   3   4   5   6   7   8   9    10  11    12
 NaN Actor NaN NaN NaN NaN NaN NaN NaN NaN  NaN NaN   NaN
Toma     1 2.0 3.0 4.0 5.0 6.0 7.0 8.0 9.0 10.0 NaN Total
   1     1 1.0 1.0 1.0 1.0 0.0 0.0 0.0 0.0  0.0 NaN     5
   2     0 0.0 1.0 1.0 1.0 0.0 0.0 0.0 0.0  0.0 NaN     3
   3     0 1.0 0.0 0.0 1.0 0.0 1.0 0.0 0.0  0.0 NaN     3

Tipos de datos:
0      object
1      object
2     float64
3     float64
4     float64
5     float64
6     float64
7     float64
8     float64
9     float64
10    float64
11    float64
12     object
dtype: object


In [57]:
# Seleccionar las filas y columnas correctas.
data_df = df.iloc[2:32, 1:11] # pandas DataFrame todavía.

# Limpiar NaN y convertir a numérico.
data_df = data_df.apply(pd.to_numeric, errors='coerce').fillna(0)

In [58]:
# Análisis de la estructura del file después de la limpieza.
print("Forma del DataFrame:", data_df.shape)
print("\nNombres exactos de las columnas:")
print(df.columns.tolist())

print("\nPrimeras 5 filas completas:")
print(data_df.head().to_string(index=False))

print("\nTipos de datos:")
print(data_df.dtypes)

Forma del DataFrame: (30, 10)

Nombres exactos de las columnas:
[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]

Primeras 5 filas completas:
 1   2   3   4   5   6   7   8   9   10
  1 1.0 1.0 1.0 1.0 0.0 0.0 0.0 0.0 0.0
  0 0.0 1.0 1.0 1.0 0.0 0.0 0.0 0.0 0.0
  0 1.0 0.0 0.0 1.0 0.0 1.0 0.0 0.0 0.0
  1 1.0 0.0 0.0 0.0 0.0 1.0 1.0 0.0 0.0
  0 1.0 0.0 1.0 0.0 0.0 0.0 1.0 0.0 0.0

Tipos de datos:
1       int64
2     float64
3     float64
4     float64
5     float64
6     float64
7     float64
8     float64
9     float64
10    float64
dtype: object


In [59]:
# Ahora sí pasar a numpy.
participacion = data_df.astype(int).to_numpy()

# Verificaciones.
print("Forma:", participacion.shape)
print("Primeras 5 filas:\n", participacion[:5])
print("Suma por actor:", participacion.sum(axis=0))

Forma: (30, 10)
Primeras 5 filas:
 [[1 1 1 1 1 0 0 0 0 0]
 [0 0 1 1 1 0 0 0 0 0]
 [0 1 0 0 1 0 1 0 0 0]
 [1 1 0 0 0 0 1 1 0 0]
 [0 1 0 1 0 0 0 1 0 0]]
Suma por actor: [22 14 13 15 11  8  3  4  2  2]


In [60]:
# Constantes.
N_TOMAS   = participacion.shape[0]     # 30
N_ACTORES = participacion.shape[1]     # 10
N_DIAS    = 5
TOMAS_POR_DIA = 6

# Precomputamos para cada actor las tomas en las que participa.
tomas_por_actor = [[] for _ in range(N_ACTORES)]
for t in range(N_TOMAS):
    for a in range(N_ACTORES):
        if participacion[t, a] == 1:
            tomas_por_actor[a].append(t)

In [61]:
# FUNCIÓN OBJETIVO: Número total de días trabajados.
def coste(sol):
    # sol = lista o array de longitud N_TOMAS --> sol[t] = día de la toma t (0..4)
    dias_trabajados = [set() for _ in range(N_ACTORES)]
    for t in range(N_TOMAS):
        d = sol[t]
        for a in range(N_ACTORES):
            if participacion[t, a]:
                dias_trabajados[a].add(d)
    return sum(len(s) for s in dias_trabajados)

In [62]:
# Generar solución inicial aleatoria (válida: exactamente 6 tomas por día).
def solucion_aleatoria():
    tomas = list(range(N_TOMAS))
    random.shuffle(tomas)
    sol = [-1] * N_TOMAS
    idx = 0
    for d in range(N_DIAS):
        for _ in range(TOMAS_POR_DIA):
            sol[tomas[idx]] = d
            idx += 1
    return sol

In [63]:
# Búsqueda local simple: Intercambiar dos tomas de días distintos.
def busqueda_local_2swap(sol):
    mejor = sol[:]
    mejor_coste = coste(mejor)

    mejora = True
    iter_sin_mejora = 0
    max_iter = 2000  # para evitar bucles infinitos en algunos casos

    while mejora and iter_sin_mejora < max_iter:
        mejora = False
        for t1 in range(N_TOMAS):
            for t2 in range(t1+1, N_TOMAS):
                d1 = sol[t1]
                d2 = sol[t2]
                if d1 == d2: continue

                # Intercambiar.
                sol[t1], sol[t2] = d2, d1
                nuevo_c = coste(sol)

                if nuevo_c < mejor_coste:
                    mejor = sol[:]
                    mejor_coste = nuevo_c
                    mejora = True
                    iter_sin_mejora = 0
                else:
                    # Deshacer.
                    sol[t1], sol[t2] = d1, d2
                iter_sin_mejora += 1

        sol[:] = mejor

    return mejor, mejor_coste

In [64]:
# Preserva la cardinalidad mediante intercambios balanceados.
def reparar_balance(sol):
    """Ajusta la solución para que cada día tenga exactamente 6 tomas"""
    conteo = [0] * N_DIAS
    for d in sol:
        conteo[d] += 1

    # Días con exceso y déficit.
    exceso = [d for d in range(N_DIAS) if conteo[d] > TOMAS_POR_DIA]
    deficit = [d for d in range(N_DIAS) if conteo[d] < TOMAS_POR_DIA]

    # Si no hay desbalance, devolver tal cual.
    if not exceso or not deficit:
        return sol

    # Intercambiar tomas entre días con exceso y déficit.
    nuevas_posiciones = list(range(N_TOMAS))
    random.shuffle(nuevas_posiciones)

    for pos in nuevas_posiciones:
        d_actual = sol[pos]
        if d_actual in exceso and deficit:
            # Elegir un día con déficit.
            d_nuevo = random.choice(deficit)
            sol[pos] = d_nuevo
            conteo[d_actual] -= 1
            conteo[d_nuevo] += 1

            # Actualizar listas.
            if conteo[d_actual] == TOMAS_POR_DIA:
                exceso.remove(d_actual)
            if conteo[d_nuevo] == TOMAS_POR_DIA:
                deficit.remove(d_nuevo)

            if not exceso or not deficit:
                break

    return sol

In [65]:
# Cruce.
def cruce_uniforme(padre1, padre2):
    """Cruce uniforme: cada posición elige aleatoriamente de uno u otro padre"""
    hijo = []
    for g1, g2 in zip(padre1, padre2):
        if random.random() < 0.5:
            hijo.append(g1)
        else:
            hijo.append(g2)
    return reparar_balance(hijo) # importante: reparar para mantener 6 por día

In [66]:
# Mutación simple.
def mutacion(sol):
    t1, t2 = random.sample(range(N_TOMAS), 2)
    while sol[t1] == sol[t2]:
        t2 = random.randint(0, N_TOMAS-1)
    sol[t1], sol[t2] = sol[t2], sol[t1]
    return sol

In [67]:
# Algoritmo Genético + mejora local = Algoritmo Memético.
def algoritmo_genetico():
    TAM_POB = 80
    GENERACIONES = 200
    PROB_CRUCE = 0.75
    PROB_MUT = 0.20
    ELITISMO = 4

    poblacion = [solucion_aleatoria() for _ in range(TAM_POB)]
    costes_pob = [coste(ind) for ind in poblacion]

    mejor_global = min(poblacion, key=coste)
    mejor_coste_global = coste(mejor_global)

    print(f"Coste inicial mejor: {mejor_coste_global}")

    for gen in range(GENERACIONES):
        nueva_pob = []

        # Elitismo.
        idx_orden = np.argsort(costes_pob)
        for i in range(ELITISMO):
            nueva_pob.append(poblacion[idx_orden[i]][:])

        while len(nueva_pob) < TAM_POB:
            cand = random.sample(range(TAM_POB), 3)
            p1 = poblacion[min(cand, key=lambda i: costes_pob[i])]
            cand = random.sample(range(TAM_POB), 3)
            p2 = poblacion[min(cand, key=lambda i: costes_pob[i])]

            if random.random() < PROB_CRUCE:
                hijo = cruce_uniforme(p1, p2)
                hijo = reparar_balance(hijo)   # siempre reparar después del cruce
            else:
                hijo = copy.deepcopy(random.choice([p1, p2]))

            if random.random() < PROB_MUT:
                hijo = mutacion(hijo)

            nueva_pob.append(hijo)

        poblacion = nueva_pob
        costes_pob = [coste(ind) for ind in poblacion]

        mejor_gen = min(poblacion, key=coste)
        c_gen = coste(mejor_gen)
        if c_gen < mejor_coste_global:
            mejor_global = mejor_gen[:]
            mejor_coste_global = c_gen
            print(f"Gen {gen:3d} → nuevo mejor: {c_gen}")

        # Mejora local cada 12 generaciones (para no ralentizar demasiado).
        if gen % 12 == 0 and gen > 0:
            mejor_local, c_local = busqueda_local_2swap(mejor_global[:])
            if c_local < mejor_coste_global:
                mejor_global = mejor_local
                mejor_coste_global = c_local
                print(f"  Mejora local → {c_local}")

    return mejor_global, mejor_coste_global

In [68]:
# Ejecución...
if __name__ == "__main__":
    random.seed(42) # para reproducibilidad en pruebas

    start = time.time()
    mejor_sol, coste_opt = algoritmo_genetico()
    end = time.time()

    print("\n" + "="*70)
    print(f"Solución encontrada (coste = {coste_opt}) en {end-start:.1f} segundos")

    # Días trabajados por cada actor.
    dias_por_actor = [len(set(mejor_sol[t] for t in tomas_por_actor[a])) for a in range(N_ACTORES)]
    print("Días trabajados por actor:", dias_por_actor)
    print("Coste total (suma):", sum(dias_por_actor))

    # Mostrar tomas por día (tomas numeradas 1 a 30).
    tomas_por_dia = [[] for _ in range(N_DIAS)]
    for t in range(N_TOMAS):
        tomas_por_dia[mejor_sol[t]].append(t+1)

    for d in range(N_DIAS):
        print(f"Día {d+1}: {sorted(tomas_por_dia[d])}")

Coste inicial mejor: 34
Gen   2 → nuevo mejor: 33
Gen   6 → nuevo mejor: 32
  Mejora local → 31
Gen  18 → nuevo mejor: 30
Gen  27 → nuevo mejor: 29
Gen 123 → nuevo mejor: 28

Solución encontrada (coste = 28) en 2.6 segundos
Días trabajados por actor: [5, 4, 3, 4, 3, 3, 1, 3, 1, 1]
Coste total (suma): 28
Día 1: [1, 10, 12, 13, 26, 29]
Día 2: [14, 17, 18, 19, 23, 24]
Día 3: [8, 9, 16, 21, 25, 28]
Día 4: [3, 4, 6, 15, 27, 30]
Día 5: [2, 5, 7, 11, 20, 22]


#Conclusiones

>En una ejecución representativa del algoritmo memético implementado, se obtuvo un coste de 28 en tan solo 2.6 segundos, partiendo de un coste inicial aleatorio de 34.

>La evolución mostró mejoras constantes: De 34 --> 28 en 123 generaciones, con una mejora significativa gracias a la búsqueda local (de 32 a 31 en una sola llamada).

>El actor principal (Actor 1) trabaja 5 días, mientras que tres actores solo asisten un día, lo que demuestra una buena minimización de convocatorias.

>Aunque el óptimo conocido para esta instancia ronda los 23-25, el valor 28 obtenido es competitivo considerando el tiempo de cómputo limitado y la ausencia de solvers exactos.

**Conclusión de calidad:**

*Estas sugerencias fueron dadas por ChatGPT.

28 es una buena solución práctica (no óptima, pero cercana).

Como ya se ha mencionado, en ejecuciones más largas (más generaciones o más búsquedas locales) este resultado suele bajar a 25-26 y quizas a 24 o incluso 23.

Sugerencias para mejorar (si se quiere bajar más el coste):

>Aumenta las generaciones o la frecuencia de búsqueda local:

- GENERACIONES = 600 # o 800

  y cambia el if de búsqueda local a:

  if gen % 10 == 0 and gen > 0:   # cada 10 (o 8) generaciones en vez de 12

>Aumenta un poco la población o el elitismo:

- TAM_POB = 150 # en vez de 80

  ELITISMO = 8  # en vez de 4

>Añade una mutación más agresiva (intercambiar 2-3 pares en vez de 1):

- def mutacion_agresiva(sol):

    for _ in range(random.randint(1,3)):

        sol = mutacion(sol)  # llama varias veces

    return sol

>Ejecuta varias veces y toma el mejor (monte-carlo simple):

- mejor_global = None

  mejor_coste = float('inf')

  for run in range(5):  # 5 ejecuciones independientes

    sol, c = algoritmo_genetico()

    if c < mejor_coste:

        mejor_coste = c
        
        mejor_global = sol

#Referencias

>Wikipedia: “Genetic algorithm”, “Memetic algorithm”, “Local search (optimization)”:

- https://en.wikipedia.org/wiki/Genetic_algorithm

- https://en.wikipedia.org/wiki/Memetic_algorithm

- https://en.wikipedia.org/wiki/Local_search_(optimization)

>Eiben, A.E. & Smith, J.E. (2015). Introduction to Evolutionary Computing (2ª ed.). Springer.

>Talbi, E.G. (2009). Metaheuristics: From Design to Implementation. Wiley.

>Caldas Lima, A. (2014). Aplicación de algoritmos heurísticos para optimizar el coste de doblaje de películas. Proyecto Fin de Máster, USC (mismo problema pero con diferentes objetivos):

- http://eio.usc.es/pub/mte/descargas/ProyectosFinMaster/Proyecto_759.pdf